In [2]:
# ========== 导入：后面要用的工具箱 ==========
# 若导入失败，请确认已在命令行激活课程虚拟环境（例如 llms）后再开内核

# 导入 os：读环境变量（Environment Variables），例如 OPENAI_API_KEY
import os
# 导入 json：把模型返回的 JSON 字符串解析成 Python 字典
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境，避免写进代码
from dotenv import load_dotenv
# 从 IPython.display 导入展示工具：用 Markdown 漂亮地显示教程结果
from IPython.display import Markdown, display, update_display
# 从本地 scraper 模块导入抓取函数：取页面链接与正文（需同目录有 scraper.py）
from scraper import fetch_website_links, fetch_website_contents
# 从 openai 导入 OpenAI 客户端：调用云端 Chat Completions API
from openai import OpenAI


In [3]:
# ========== 环境初始化 + 模型常量 ==========

# override=True：.env 里的值覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境读取 OpenAI API Key（不要把真实密钥写进笔记本）
api_key = os.getenv('OPENAI_API_KEY')

# 粗检密钥形态：以 sk-proj- 开头且长度够长，通常表示已配置（非严格校验）
if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    # 失败提示文案保持英文原样（与课程排错笔记本一致）
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# 本笔记本前半段筛选链接时使用的模型名
MODEL = 'gpt-5-nano'
# 创建 OpenAI 客户端：默认自动读 OPENAI_API_KEY
openai = OpenAI()


API key looks good so far


In [4]:
# ========== 试抓：列出 Figma 首页上的链接 ==========

# 调用本地 scraper：返回页面上的链接列表（可能含相对路径 / 锚点）
links = fetch_website_links("https://www.figma.com")
# 在笔记本里直接展示列表，方便肉眼扫一眼结构
links


['#main',
 'https://www.figma.com/',
 'https://www.figma.com/pricing/',
 'https://www.figma.com/login',
 'https://www.figma.com/contact/',
 'https://www.figma.com/signup',
 'https://www.figma.com/signup',
 'https://www.figma.com/design/',
 'https://www.figma.com/dev-mode/',
 'https://www.figma.com/figjam/',
 'https://www.figma.com/slides/',
 'https://www.figma.com/draw/',
 'https://www.figma.com/buzz/',
 'https://www.figma.com/sites/',
 'https://www.figma.com/make/',
 'https://www.figma.com/ai/',
 'https://www.figma.com/mcp-catalog/',
 'https://www.figma.com/downloads/',
 'https://www.figma.com/release-notes/',
 'https://www.figma.com/design-systems/',
 'https://www.figma.com/prototyping/',
 'https://www.figma.com/ux-design-tool/',
 'https://www.figma.com/web-design/',
 'https://www.figma.com/wireframe-tool/',
 'https://www.figma.com/figjam/online-whiteboard/',
 'https://www.figma.com/figjam/agile-workflows/',
 'https://www.figma.com/figjam/strategic-planning/',
 'https://www.figma.com

In [5]:
# ========== 链接筛选：system prompt（保持英文，决定「相关」标准） ==========

# 告诉模型：从链接里挑出适合做 knowledgebase / how-to / tutorials 的页面，并以 JSON 返回
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to build a knowledgebase/how-to/tutorials section on Figma.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [6]:
# ========== 组装 user prompt：把「要分析的网站 + 全量链接」交给模型 ==========

def get_links_user_prompt(url):
    # 先写任务说明：要挑教程/知识库相关链接，排除 ToS/隐私/邮箱等
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant to build a knowledgebase/how-to/tutorials section on Figma, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links. Do necessary translation to understand what is what if the website not in English. Mind that tutorials might be included on a separate subpage of the main website, broken down into granular how-to articles. If so, find and include rather them than generic website pages. 

Links (some might be relative links):

"""
    # 再次抓取该站链接列表（与上面试抓同一函数）
    links = fetch_website_links(url)
    # 把每条链接追加到 prompt 末尾，一行一个
    user_prompt += "\n".join(links)
    return user_prompt


In [7]:
# ========== 调用 LLM：强制 JSON 对象，选出相关链接 ==========

def select_relevant_links(url):
    # Chat Completions：system 定规则，user 给链接清单
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        # response_format：要求返回合法 JSON 对象（不是自由文本）
        response_format={"type": "json_object"}
    )
    # 取出模型回复的文本内容
    result = response.choices[0].message.content
    # 解析成 Python dict，形如 {"links": [{"type":..., "url":...}, ...]}
    links = json.loads(result)
    return links


In [8]:
# ========== 试跑：对 Figma 首页做一次「相关链接」筛选 ==========

# 应返回 JSON 解析后的字典；可对照是否偏向 help/resource/education 类页面
select_relevant_links("https://www.figma.com")


{'links': [{'type': 'knowledge base',
   'url': 'https://help.figma.com/hc/en-us'},
  {'type': 'support request',
   'url': 'https://help.figma.com/hc/en-us/requests/new'},
  {'type': 'resource library',
   'url': 'https://www.figma.com/resource-library/'},
  {'type': 'best practices', 'url': 'https://www.figma.com/best-practices/'},
  {'type': 'education resources', 'url': 'https://www.figma.com/education/'}]}

In [9]:
# ========== 汇总素材：首页正文 + 每个相关链接页的正文 ==========

def fetch_page_and_all_relevant_links(url):
    # 抓取落地页纯文本/正文
    contents = fetch_website_contents(url)
    # LLM 选出相关链接字典
    relevant_links = select_relevant_links(url)
    # 用 Markdown 小标题拼成一份大文档，方便后面再喂给模型
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    # 逐个相关链接：写类型标题，再抓该页内容拼进去
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result


In [10]:
# ========== 试跑：打印「首页 + 相关页」拼好的长文本 ==========

# 内容可能很长；主要用于检查抓取与拼接是否正常
print(fetch_page_and_all_relevant_links("https://www.figma.com"))


## Landing Page:

Figma: The Collaborative Interface Design Tool

Skip to main content
Products
Solutions
Community
Resources
Pricing
Log in
Contact sales
Get started
Get started for free
Figma Design
Design and prototype in one place
Dev Mode
Translate designs into code
FigJam
Collaborate with a digital whiteboard
Figma Slides
Co-create presentations
Figma Draw
New
Illustrate with advanced vector tools
Figma Buzz
Beta
Produce on-brand assets at scale
Figma Sites
Beta
Publish fully responsive websites
Figma Make
New
Prompt to code anything you can imagine
AI
Explore all Figma AI features
MCP
Connect Figma to AI coding tools
Downloads
Get the desktop, mobile, and font installer apps
Release Notes
See the latest features and releases
Use cases
Design systems
Prototyping
UX design
Web design
Wireframing
Online whiteboard
Agile
Strategic planning
Brainstorming
Diagramming
Product development
Web development
Design handoff
See all solutions
Roles
Design
Engineering
Product managers
Organiza

In [11]:
# ========== 教程生成：system prompt（保持英文）+ 可选幽默版注释 ==========

# 要求：按 beginner / intermediate / pro 分组；强调 AI 相关文章；禁止编造 URL
tutorial_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a well-organized, clean and informative list of tutorials/how-to pages that relate to Figma. 
The tutorial articles should be grouped in a few buckets depending on level of those who'd be undertaking learning (levels being - beginner, intermediate, pro). No need to list the entire text of the articles - titles and descriptions will do.
Apply judgement to define the levels. Make sure to include and stylistically emphasize any articles on AI. Include the links to the given tutorials in your final response. 
CRITICAL: You must ONLY use URLs that are explicitly provided in the user's message. 
    Do NOT invent, guess, or construct any URLs. If you need a URL for a topic but it 
    wasn't provided, either:
    1. Skip that link entirely, OR
    2. Mention the topic without a link
    
Never include placeholder URLs like 'XXX' or assume URL patterns.
Respond in markdown without code blocks. Formulate the wording in an easy-to-understand way and engaging, playful manner.
"""

# 或者取消注释下面的行以获得更幽默的小册子——这展示了改「语气（tone）」有多容易：

# 宣传册系统提示="""
# 您是一名助理，负责分析公司网站上几个相关页面的内容
# 并为潜在客户、投资者和新员工制作了一本关于公司的简短、幽默、有趣、诙谐的宣传册。
# 在没有代码块的情况下以 Markdown 方式进行响应。
# 如果您有信息，请包括公司文化、客户和职业/工作的详细信息。
# """


In [12]:
# ========== 组装教程任务的 user prompt：列出「唯一合法 URL」清单 ==========

def get_tutorial_user_prompt(company_name, url, links, contents):
    """构造用户消息：明确可用 URL，降低模型幻觉出假链接的概率。"""
    return f"""Create a comprehensive tutorial guide for {company_name} based on {url}.

AVAILABLE URLS (these are the ONLY valid URLs you can use):
{chr(10).join(f"- {link}" for link in links)}

WEBSITE CONTENT:
{contents}

Create an engaging, well-structured tutorial organized by skill level.
Remember: Only use URLs from the "AVAILABLE URLS" list above. 
If you want to mention a topic but don't have a valid URL for it, just describe it without a link."""


In [13]:
# ========== 后处理：校验回答里的 Markdown 链接是否在「合法 URL」集合中 ==========

# 导入正则：用来匹配 Markdown 链接写法 [text](url)
import re

def validate_urls_in_response(response_text, valid_urls):
    """
    检查 AI 回复里的 Markdown 链接是否都在 valid_urls 中。
    非法/占位 URL 会去掉超链接，只保留加粗文本，避免点到假链。
    """
    # 匹配 [锚文本](url) 形式
    url_pattern = r'\[([^\]]+)\]\(([^\)]+)\)'

    def check_and_fix_link(match):
        # group(1)=显示文字，group(2)=链接地址
        link_text = match.group(1)
        url = match.group(2)

        # URL 在有效列表中：原样保留整个 Markdown 链接
        if url in valid_urls:
            return match.group(0)
        # 占位符 URL（含 XXX）：删掉链接，文字加粗保留
        elif 'XXX' in url or 'XXXX' in url:
            return f"**{link_text}**"
        # 其它不在列表中的 URL：同样降级为加粗文本
        else:
            return f"**{link_text}**"

    # 对全文所有 Markdown 链接做替换
    fixed_response = re.sub(url_pattern, check_and_fix_link, response_text)
    return fixed_response


In [14]:
# ========== 端到端：抓取 → 选链 → 生成教程 → 校验 URL → Markdown 展示 ==========

def create_tutorial(company_name, url):
    # 抓取落地页正文
    contents = fetch_website_contents(url)
    # LLM 选出相关链接（带 type 字段的字典列表）
    relevant_links_dict = select_relevant_links(url)

    # 只抽出 URL 字符串列表，后面写进 AVAILABLE URLS
    links = [link['url'] for link in relevant_links_dict['links']]

    # 拼接「落地页 + 各相关页」大文本，作为模型阅读素材
    all_contents = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link_info in relevant_links_dict['links']:
        all_contents += f"\n\n### Link: {link_info['type']}\n"
        all_contents += fetch_website_contents(link_info['url'])

    # 合法 URL 集合：相关链接 + 入口 url
    valid_urls = set(links)
    valid_urls.add(url)

    # 再从抓取正文里用正则捞出出现过的 http(s) 链接，扩大白名单
    import re
    url_pattern = r'https?://[^\s\)\]\'"<>]+'
    for match in re.finditer(url_pattern, all_contents):
        valid_urls.add(match.group(0))

    # 打印白名单规模，便于调试「为什么链接被剥掉」
    print(f"Found {len(valid_urls)} valid URLs from scraped content")

    # 构造 user 消息；正文截断到 5000 字符，控制 prompt 体积/费用
    user_content = get_tutorial_user_prompt(company_name, url, links, all_contents[:5000])

    # 调用模型生成教程（这里硬编码 gpt-4.1-mini，与前面 MODEL 常量不同）
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": tutorial_system_prompt},
            {"role": "user", "content": user_content}
        ],
    )

    # 取出生成的 Markdown 文本
    result = response.choices[0].message.content

    # 展示前先校验/修复非法链接，再渲染
    validated_result = validate_urls_in_response(result, valid_urls)

    display(Markdown(validated_result))


In [15]:
# ========== 运行：为 Figma 生成分级教程指南 ==========

# 公司名用于文案标题；URL 是抓取与选链的入口
create_tutorial("Figma", "https://www.figma.com")


Found 4 valid URLs from scraped content


# Ultimate Figma Tutorial Guide — From Newbie to Pro (With a Dash of AI Magic!)

Hey there, future Figma master! Whether you’re just getting your feet wet or ready to dive into the advanced seas of design, prototyping, and AI-powered creativity, this guide is made just for you. We’ve scoured Figma’s treasure trove of resources to organize tutorials and how-to’s into easy buckets: Beginner, Intermediate, and Pro. Let’s get designing!

---

## Beginner Level: Getting Your Figma Feet Wet

### 1. Figma Learn - Help Center  
**Start here if you’re new to Figma!** Explore the basics of Figma Design, prototyping, and collaboration. Learn how to navigate files, projects, and the workspace essentials all in one spot.  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)

### 2. Introduction to Figma Design and Prototyping  
Learn how to create your first design and make it interactive with prototyping tools. This foundation will have you quickly visualizing ideas and sharing them.  
Available through the Help Center’s "Product documentation" and "Courses, tutorials, projects" sections.  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)

### 3. Explore Figma’s Collaborative Whiteboard - FigJam  
Get to know FigJam, Figma's digital whiteboard, perfect for brainstorming and team collaboration. Fun and interactive starting point!  
🔗 [Figma Learn - Help Center - FigJam](https://help.figma.com/hc/en-us)

---

## Intermediate Level: Level Up Your Design and Workflow

### 1. Resource Library — How-To & Best Practice Articles  
Now that you’ve got your fundamentals, dive into best practices for design systems, UX, web design, wireframing, and more. These guides help you sharpen your skills and create polished projects.  
📰 Explore smart workflows, prototyping strategies, and team collaboration tricks here:  
🔗 [Resource Library](https://www.figma.com/resource-library/)

### 2. Design Hand-off & Dev Mode  
Learn how to efficiently translate your designs to developers with Figma’s Dev Mode. Understand how to organize specs, code snippets, and assets for smooth handoff.  
Available in both the Help Center and Resource Library under design-to-development workflows.  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)  
🔗 [Resource Library](https://www.figma.com/resource-library/)

### 3. Using Figma Slides for Presentations  
Start co-creating presentations directly in Figma. This boosts your workflow by keeping design and presentation in one place.  
Details can be found in Help Center’s tutorials on Figma Slides.  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)

---

## Pro Level: Mastery, Automation, and AI-Powered Design

### 1. Explore All Figma AI Features — **The Future is Now!**  
Figma’s AI tools let you prompt, design, and generate code like a wizard of design sorcery. This is where you blend creativity and efficiency with AI assistance to bring ideas to life faster than ever.  
Find comprehensive guides and feature explanations in both the Help Center and Resource Library’s AI sections.  
⭐ Important: If you’re keen on AI in design, this is your *go-to* tutorial area!  
🔗 [Figma Learn - Help Center (AI)](https://help.figma.com/hc/en-us)  
🔗 [Resource Library - AI](https://www.figma.com/resource-library/)

### 2. Figma Make — Prompt to Code Anything You Imagine  
Harness AI to instantly generate code and design from simple prompts. This tutorial digs deep into Figma Make’s potential for developers and designers who want to automate and innovate.  
Find this cutting-edge content in the Resource Library or Help Center under Figma Make and AI features.  
🔗 [Resource Library](https://www.figma.com/resource-library/)  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)

### 3. Advanced Vector Illustration with Figma Draw  
Ready to go beyond basic shapes? Learn to illustrate with Figma Draw, the advanced vector toolset that lets pros create stunning, scalable art directly in Figma.  
Hint: This is great for UI artists and complex icon designers.  
Check tutorials in the Help Center’s Figma Draw section.  
🔗 [Figma Learn - Help Center](https://help.figma.com/hc/en-us)

### 4. Building Responsive Websites with Figma Sites (Beta)  
For the pros who want to publish fully responsive websites directly from Figma — this guide walks you through the workflow, tips, and best practices to launch your site faster.  
No direct tutorial link, but check Figma Sites info on Help Center or Resource Library for updates.

---

## Bonus: Education and Hands-on Learning

### Figma Education  
If you’re a student, teacher, or just love structured learning, Figma Education offers courses and projects tailored to skill-building in design. It’s a fantastic place to get certified and build your portfolio.  
🔗 [Figma Education](https://www.figma.com/education/)

---

# Ready to Dive In?

Here’s your launchpad again to explore tutorials on Figma’s official education and help sections:

- **Help Center (All levels, foundational to pro AI & dev topics):** https://help.figma.com/hc/en-us  
- **Resource Library (Tutorials & Best Practice Articles):** https://www.figma.com/resource-library/  
- **Education (Courses, projects for students & teachers):** https://www.figma.com/education/

---

Happy designing, prototyping, jamming, and pushing the limits of what Figma and AI can do! 🚀✨

In [ ]:
# （空单元格占位：原作者未写内容；需要实验时可在此继续扩展）
